# 〰️ Beyond Linearity
**ISLP Chapter 7 · Pattern Recognition for the Rest of Us**

> Linear regression assumes a straight-line relationship. But many real-world relationships are curved. This notebook covers flexible extensions: polynomial regression, splines, and Generalized Additive Models — all interpretable ways to model nonlinearity.

### What you'll learn
- Polynomial regression: fitting curves with OLS
- Step functions: piecewise constants
- Regression splines: smooth curves with knots
- Natural cubic splines
- GAMs: additive models with one smooth term per feature

### Dataset: Wage (ISLP) + Auto
### Time: ~55 minutes

In [ ]:
# Overview: income vs age relationship
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(wage['age'], wage['wage'], alpha=0.15, color='#888', s=10)
ax.set_xlabel('Age'); ax.set_ylabel('Wage ($)')
ax.set_title('Wage vs Age — clearly nonlinear, needs more than a straight line')
plt.tight_layout(); plt.show()

## 📐 Part 1 — Polynomial Regression

Extend linear regression with polynomial terms:
```
Y = β₀ + β₁X + β₂X² + β₃X³ + ... + βdXd + ε
```
This is still linear regression (linear in the β coefficients) — OLS works perfectly. The trick is just creating the polynomial features from X.

In [ ]:
# Polynomial regression: compare degrees
X_age = wage['age'].values.reshape(-1,1)
y_wage = wage['wage'].values
age_range = np.linspace(wage['age'].min(), wage['age'].max(), 200).reshape(-1,1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
degrees = [1, 2, 4, 10]
for ax, d in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#1e5fa8', lw=2.5)
    ax.set_title(f'Degree {d}\n10-fold CV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Polynomial Regression: Increasing Degree', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Degree 4 captures the curve. Degree 10 wiggles — overfitting at the extremes')

## 🪢 Part 2 — Splines: Smooth Piecewise Polynomials

Polynomials are global — a wiggle at one end affects predictions everywhere. **Splines** are piecewise polynomials that join smoothly at **knots**.

A **regression spline** with K knots has K+d+1 basis functions (d = degree). The result: flexibility where the data is dense, smoothness everywhere.

**Natural cubic splines** add the constraint that the function is linear beyond the boundary knots — preventing wild behavior at the extremes.

In [ ]:
# Compare: linear, polynomial, spline
from sklearn.preprocessing import SplineTransformer

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_spline = [
    ('Degree-4 Poly', Pipeline([('poly',PolynomialFeatures(4)),('lr',LinearRegression())])),
    ('Spline (4 knots)', Pipeline([('spline',SplineTransformer(n_knots=4, degree=3)),('lr',LinearRegression())])),
    ('Spline (8 knots)', Pipeline([('spline',SplineTransformer(n_knots=8, degree=3)),('lr',LinearRegression())])),
]

for ax, (title, pipe) in zip(axes, models_spline):
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#e85d20', lw=2.5)
    ax.set_title(f'{title}\nCV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Splines vs Polynomial', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## ➕ Part 3 — Generalized Additive Models (GAMs)

GAMs extend linear models to allow nonlinear relationships:
```
Y = β₀ + f₁(X₁) + f₂(X₂) + ... + fₚ(Xₚ) + ε
```
Each fⱼ is a smooth function estimated from the data. The **additive** structure means:
- Each feature contributes independently
- Interpretable: plot each fⱼ to see its effect
- Much more flexible than linear but more interpretable than neural networks

In [ ]:
!pip install pygam -q
from pygam import LinearGAM, s, f, l

# GAM: wage ~ s(age) + s(year) + education
wage_enc = pd.get_dummies(wage[['age','year','education','wage']], drop_first=True, dtype=float)
feature_cols = [c for c in wage_enc.columns if c != 'wage']
X_gam = wage_enc[feature_cols].values
y_gam = wage_enc['wage'].values

gam = LinearGAM(s(0) + s(1) + l(2) + l(3) + l(4) + l(5)).fit(X_gam, y_gam)
print(f'GAM R²: {gam.statistics_["pseudo_r2"]["McFadden"]:.4f}')
gam.summary()

In [ ]:
# Plot GAM smooth terms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, term_idx, label in zip(axes, [0, 1], ['age', 'year']):
    XX = gam.generate_X_grid(term=term_idx)
    pdep, confi = gam.partial_dependence(term=term_idx, width=0.95)
    ax.plot(XX[:,term_idx], pdep, color='#1e5fa8', lw=2.5)
    ax.fill_between(XX[:,term_idx], confi[:,0], confi[:,1], alpha=0.2, color='#1e5fa8')
    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel(label); ax.set_ylabel(f'Effect on wage')
    ax.set_title(f'GAM: smooth term for {label}')
plt.suptitle('GAM Partial Effects — Age and Year', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Wage rises steeply with age until ~40, then levels off')
print('   Year shows a gradual upward trend — wages rising over time')

In [ ]:
answers = {
    'q1': '',  # What is a regression spline and how does it differ from a polynomial?
    'q2': '',  # What are knots in the context of splines?
    'q3': '',  # What is the additive assumption in GAMs?
    'q4': '',  # How do you choose the number/location of knots?
    'q5': '',  # Name one advantage of splines over high-degree polynomials
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Beyond Linearity"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
